# 2장 실습 — 도로망 데이터 열어 보기

0장 결과의 평균 대기시간 4.1분은 차가 도로를 따라 승객에게 가는 시간을 더한 값입니다.
그 계산에 쓰인 도로망을 직접 엽니다. 교재 2장 전체에 대응합니다.

도로망은 표 두 개, 노드 표와 엣지 표입니다.
두 표를 읽고, 엣지의 양 끝 노드를 찾고, 인접 리스트로 바꾸고, 좌표를 노드에 붙입니다.
여기까지 되면 3장에서 최단경로를 구할 준비가 끝납니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 표 두 개 (교재 2.1)

하남시 도로망은 parquet 파일 두 개입니다. `read_parquet` 으로 읽으면 pandas 데이터프레임이 됩니다.
`shape` 는 (행 수, 컬럼 수)입니다. 엣지 표에서는 이 장에서 쓰는 컬럼 여섯 개만 골라 앞 다섯 줄을 봅니다.

In [ ]:
import pandas as pd

from smartmob.data import data_path

nodes = pd.read_parquet(data_path("hanam/road_graph_nodes.parquet"))
edges = pd.read_parquet(data_path("hanam/road_graph_edges.parquet"))

print("노드 표", nodes.shape)
print("엣지 표", edges.shape)
edges[["edge_id", "highway", "name", "length", "free_flow_speed_kmh", "oneway"]].head()

노드 21,931개, 엣지 59,873개입니다. 0장에서 본 12,566개와 28,589개보다 훨씬 많습니다.
0장의 값은 자동차가 다닐 수 있는 도로만 남긴 뒤의 것이고, 여기서는 거르기 전의 표를 봅니다.

엣지 하나가 교차로와 교차로 사이의 도로 한 토막입니다.
`length` 는 미터, `free_flow_speed_kmh` 는 막히지 않을 때의 속도입니다.
둘을 나누면 그 토막을 지나는 데 걸리는 초가 나옵니다.

## 2. 양 끝 노드는 어디에 있는가 (교재 2.2)

엣지 표에 출발 노드와 도착 노드 컬럼이 없습니다. 대신 `edge_id` 안에 들어 있습니다.
`e37375263_f_445273230_436257996` 은 네 조각입니다.
도로(way) 번호, 방향(`f` 정방향·`r` 역방향), 출발 노드의 OSM 번호, 도착 노드의 OSM 번호입니다.
노드 표의 `node_id` 는 OSM 번호 앞에 `n` 을 붙인 것이므로, 뒤의 두 조각에 `n` 을 붙이면 양 끝 노드가 됩니다.

In [ ]:
def parse_edge_id(edge_id):
    # rsplit("_", 2) 는 오른쪽에서부터 "_" 를 두 번만 잘라 세 조각을 만듭니다.
    # 도로 번호와 방향이 붙은 앞부분은 통째로 남고, 뒤의 두 조각이 출발·도착 노드입니다.
    _, source_osm, target_osm = edge_id.rsplit("_", 2)
    return f"n{source_osm}", f"n{target_osm}"


print(parse_edge_id("e37375263_f_445273230_436257996"))
print("방향:", edges["direction"].value_counts().to_dict())
print("일방통행:", edges["oneway"].value_counts().to_dict())

양방향 도로는 `f` 줄과 `r` 줄이 하나씩 들어 있고, 일방통행은 2,413개뿐입니다.
그래서 각 줄을 그냥 단방향 엣지로 다루면 됩니다.
파싱 규칙이 맞는지 `osm_node_seq_json` 과 대조하는 확인은 교재 2.2절에 있습니다.

## 3. 엣지의 절반은 자동차가 못 다닙니다 (교재 2.3)

`highway` 가 도로의 종류입니다. `value_counts()` 는 값별로 몇 줄인지 세어 많은 순으로 늘어놓습니다.

In [ ]:
top = edges["highway"].value_counts().head(10)
top

가장 많은 것이 `footway`, 즉 보도로 22,064개입니다. `cycleway` 와 `path` 까지 더하면 전체의 절반이 넘습니다.
택시 시뮬레이션에 보도를 넣으면 차가 인도로 달립니다. 그래서 통행수단에 맞는 `highway` 만 남겨야 합니다.

`load_road_graph` 의 `modes` 인자가 그 필터입니다. `("walk",)` 로 바꾸면 보행망이 나옵니다.
엣지 표의 속도 컬럼은 자동차 기준이라 사람이 걷는 속도로는 쓸 수 없습니다.
그래서 보행망에는 `speed_kmh=5` 로 속도를 직접 줍니다.

In [ ]:
from smartmob.data import load_road_graph

drive = load_road_graph("hanam", modes=("drive",))
walk = load_road_graph("hanam", modes=("walk",), speed_kmh=5)    # 보행망. 모든 엣지가 5km/h

banner("modes 필터의 효과")
expect("자동차 도로 노드", drive.n_nodes, 12_566)
expect("자동차 도로 엣지", drive.n_edges, 28_589)
print(f"보행망: 노드 {walk.n_nodes:,}, 엣지 {walk.n_edges:,}")
print(f"자동차가 쓸 수 있는 엣지는 표 전체 {len(edges):,}개의 {drive.n_edges / len(edges):.0%} 입니다")

자동차 도로망은 노드 12,566개, 엣지 28,589개로 0장과 같습니다.
보행망은 노드 20,526개, 엣지 49,313개로 더 큽니다. 표 전체 59,873개 중 자동차가 쓸 수 있는 엣지는 48%입니다.

## 4. 인접 리스트 (교재 2.4)

최단경로를 구하려면 "이 노드에서 갈 수 있는 곳"을 빠르게 답해야 합니다.
표를 매번 훑는 대신 노드마다 나가는 엣지 목록을 미리 만들어 둔 것이 인접 리스트입니다.
`RoadGraph` 가 읽을 때 이미 만들어 둡니다.
`neighbors(노드)` 는 (이웃 노드, 소요시간 초, 엣지 번호) 세 값을 돌려줍니다.

In [ ]:
# next(...) 는 조건에 맞는 첫 번째 노드 하나만 꺼냅니다. 이웃이 셋 이상인 교차로를 고릅니다.
sample = next(n for n in drive.nodes if len(drive.adj[n]) >= 3)

banner(f"노드 {sample} 의 이웃")
for neighbour, seconds, edge_index in drive.neighbors(sample):
    print(f"  → {neighbour}  {seconds:6.1f}초  (엣지 {edge_index})")

값이 거리가 아니라 **초**입니다.
200m짜리 주택가 도로(30km/h)와 200m짜리 간선도로(60km/h)는 거리가 같아도 시간이 두 배 차이 납니다.
시뮬레이터가 알고 싶은 것은 시간입니다.
엣지 번호는 엣지 표의 행 번호이고, 4장에서 경로가 지나는 도로의 종류를 찾아볼 때 씁니다.

## 5. 좌표로 노드 찾기 (교재 2.5)

승객은 노드 위에서 택시를 부르지 않고 아무 좌표에서나 부릅니다.
그 좌표를 가장 가까운 노드로 옮기는 것을 스냅(snapping)이라고 합니다.
`nearest_node` 는 노드 좌표를 KD-트리에 한 번 넣어 두고 찾으므로 1,000건도 순식간에 끝납니다.
`haversine_km` 은 위경도 두 쌍 사이의 직선거리를 km 로 돌려줍니다.

In [ ]:
from smartmob.teaching.graph import haversine_km

HANAM_CITY_HALL = (37.5393, 127.2148)
MISA_STATION = (37.5606, 127.1930)

start = drive.nearest_node(*HANAM_CITY_HALL)    # * 는 튜플을 (위도, 경도) 두 인자로 풀어 줍니다
goal = drive.nearest_node(*MISA_STATION)

print("하남시청 →", start, drive.coord[start])
print("미사역   →", goal, drive.coord[goal])
print(f"두 지점 직선거리 {haversine_km(*HANAM_CITY_HALL, *MISA_STATION):.2f} km")

스냅된 노드의 좌표가 입력한 좌표와 소수점 셋째 자리까지 같습니다. 수십 미터 안의 노드에 붙은 것입니다.
직선거리는 3.05km입니다. 3장에서 이 두 노드 사이의 도로 경로를 구하면 4.2km가 나옵니다.

## 6. 빈칸 (교재 2.6)

### 6.1 가장 긴 엣지

자동차 도로 중 `length` 가 가장 긴 엣지 다섯 개를 찾고, 그 도로의 이름과 종류를 봅니다.
`nlargest(5, "length")` 가 그 다섯 행을 돌려줍니다. 왜 이런 엣지가 길게 나오는지 한 줄로 적습니다.

In [ ]:
drive_edges = drive.edges       # modes 필터를 통과한 엣지만 남은 표
print("자동차 도로 엣지 표", drive_edges.shape)

longest = None      # drive_edges 에서 length 상위 5개 행 (DataFrame)

banner("빈칸 6.1")
todo("가장 긴 엣지 5개", longest, fmt=lambda df: f"{len(df)}행, 최장 {df['length'].max():.0f}m")

### 6.2 속도가 0인 엣지

`free_flow_speed_kmh` 가 0이거나 비어 있는 엣지가 있으면 소요시간이 무한대가 됩니다.
자동차 도로망에 그런 엣지가 몇 개 있는지 셉니다. `isna()` 가 결측을, `<= 0` 이 0 이하를 잡습니다.
0개가 나오면 그것이 답입니다.

In [ ]:
bad_speed = None    # 속도가 0이거나 결측인 자동차 도로 엣지 수 (정수)

banner("빈칸 6.2")
todo("속도가 없는 엣지", bad_speed)

### 6.3 한 방향만 뚫린 길

`oneway` 가 참인 엣지의 비율을 구합니다. 참·거짓 컬럼의 `mean()` 이 곧 참인 비율입니다.
일방통행을 무시하고 최단경로를 구하면 무엇이 잘못되는지 한 줄로 적습니다.

In [ ]:
oneway_share = None     # 일방통행 엣지의 비율 (0~1)

banner("빈칸 6.3")
todo("일방통행 비율", oneway_share, fmt=lambda v: f"{v:.1%}")

## 정리

- 도로망은 노드 표와 엣지 표입니다. `RoadGraph` 가 둘을 읽어 인접 리스트로 바꿉니다
- 엣지의 양 끝 노드는 `edge_id` 안에 있고 `rsplit("_", 2)` 로 꺼냅니다
- `modes` 로 거르지 않으면 택시가 보도로 달립니다. 표 전체의 48%만 자동차 도로입니다
- 엣지의 비용은 거리가 아니라 초입니다
- `nearest_node` 가 위경도를 노드에 붙입니다
- 3장 실습에서는 이 인접 리스트 위에서 최단경로를 직접 짭니다